<a href="https://colab.research.google.com/github/shivanvithajayam/1BF24CS127_AI_LAB/blob/main/Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

# ==========================================
# 1. PARSER: Converts string input into list structure
# ==========================================
def parse_expression(expr_str):
    """
    Parses a string like "P(x, f(a))" into a structured list: ['P', 'x', ['f', 'a']].
    """
    expr_str = expr_str.replace(" ", "")  # Remove whitespaces
    tokens = re.split(r'([(),])', expr_str)
    tokens = [t for t in tokens if t]     # Clear out empty strings

    def parse_tokens(token_list):
        if not token_list:
            return None

        token = token_list.pop(0)

        # If it's followed by a open parenthesis, it's a predicate or a function
        if token_list and token_list[0] == '(':
            token_list.pop(0)  # Remove '('
            args = []
            while token_list and token_list[0] != ')':
                arg = parse_tokens(token_list)
                if arg:
                    args.append(arg)
                if token_list and token_list[0] == ',':
                    token_list.pop(0)  # Remove ','
            if token_list:
                token_list.pop(0)  # Remove ')'
            return [token] + args
        else:
            return token

    return parse_tokens(tokens)

# ==========================================
# 2. LOGIC HELPER FUNCTIONS
# ==========================================
def is_variable(x):
    """Variables are single lowercase letters (e.g., x, y, z)."""
    return isinstance(x, str) and x.islower() and len(x) == 1

def is_constant(x):
    """Constants are single uppercase letters (e.g., A, B, C)."""
    return isinstance(x, str) and x.isupper() and len(x) == 1

def is_function(x):
    """Functions are lists where the head name is lowercase (e.g., ['f', 'x'])."""
    return isinstance(x, list) and isinstance(x[0], str) and x[0].islower()

def occurs_in(var, expr):
    """Checks if a variable occurs inside a functional term or expression."""
    if var == expr:
        return True
    if isinstance(expr, list):
        # Skip the predicate/function name at index 0, check arguments
        return any(occurs_in(var, sub_expr) for sub_expr in expr[1:])
    return False

def apply_substitution(subst, expr):
    """Applies a substitution dictionary mapping to an entire structural term."""
    if not subst:
        return expr
    if isinstance(expr, str):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        # Keep name, apply to args recursively
        return [expr[0]] + [apply_substitution(subst, sub) for sub in expr[1:]]
    return expr

def format_expression(expr):
    """Converts a structural list back into an easily readable logic string."""
    if isinstance(expr, str):
        return expr
    if isinstance(expr, list):
        return f"{expr[0]}({', '.join(format_expression(arg) for arg in expr[1:])})"
    return str(expr)

# ==========================================
# 3. UNIFICATION ALGORITHM
# ==========================================
def unify(psi1, psi2):
    # Step 1: If psi1 or psi2 is a variable or constant
    # (Note: Functions like f(x) are lists, so they fall through to Step 2/3/5)
    if isinstance(psi1, str) or isinstance(psi2, str):
        if psi1 == psi2:
            return {}

        elif is_variable(psi1):
            if occurs_in(psi1, psi2):
                return "FAILURE"
            else:
                return {psi1: psi2}

        elif is_variable(psi2):
            if occurs_in(psi2, psi1):
                return "FAILURE"
            else:
                return {psi2: psi1}

        else:
            return "FAILURE"

    # Step 2: Check matching symbol heads (Predicate/Function names)
    if psi1[0] != psi2[0]:
        return "FAILURE"

    # Step 3: Match argument structural lengths
    if len(psi1) != len(psi2):
        return "FAILURE"

    # Step 4: Set Substitution set (SUBST) to NIL
    subst = {}

    # Step 5: Process arguments sequentially
    args1 = psi1[1:]
    args2 = psi2[1:]

    for i in range(len(args1)):
        s = unify(args1[i], args2[i])

        if s == "FAILURE":
            return "FAILURE"

        if s != {}:
            args1 = [apply_substitution(s, arg) for arg in args1]
            args2 = [apply_substitution(s, arg) for arg in args2]

            # Compose substitution tracking
            for var in subst:
                subst[var] = apply_substitution(s, subst[var])
            subst.update(s)

    return subst

# ==========================================
# 4. USER INTERACTION INTERFACE
# ==========================================
if __name__ == "__main__":
    print("--- First-Order Logic Unification System ---")
    print("Rules: Uppercase (A, B) = Constants | Lowercase (x, y) = Variables")
    print("Examples: 'P(A, B, B)' and 'P(x, y, z)' OR 'P(x, f(x))' and 'P(A, f(A))'\n")

    input1 = input("Enter Expression 1 (Ψ1): ").strip()
    input2 = input("Enter Expression 2 (Ψ2): ").strip()

    try:
        expr1 = parse_expression(input1)
        expr2 = parse_expression(input2)

        if not expr1 or not expr2 or not isinstance(expr1, list) or not isinstance(expr2, list):
            raise ValueError("Invalid format. Make sure to use standard predicate form like P(...)")

        result = unify(expr1, expr2)

        print("\n--- Results ---")
        print(f"Parsed Ψ1: {format_expression(expr1)}")
        print(f"Parsed Ψ2: {format_expression(expr2)}")
        print("Substitution Set (SUBST):")

        if result == "FAILURE":
            print("Unification Failed (Mismatched symbols or Occur Check failed).")
        elif result == {}:
            print("NIL (Expressions are already structurally identical)")
        else:
            formatted_subst = ", ".join([f"({format_expression(val)}/{key})" for key, val in result.items()])
            print(f"{{ {formatted_subst} }}")

    except Exception as e:
        print(f"\nParsing Error: Please check your syntax format. ({e})")


--- First-Order Logic Unification System ---
Rules: Uppercase (A, B) = Constants | Lowercase (x, y) = Variables
Examples: 'P(A, B, B)' and 'P(x, y, z)' OR 'P(x, f(x))' and 'P(A, f(A))'

Enter Expression 1 (Ψ1): P(f(a),g(Y))
Enter Expression 2 (Ψ2): P(X,X)

--- Results ---
Parsed Ψ1: P(f(a), g(Y))
Parsed Ψ2: P(X, X)
Substitution Set (SUBST):
Unification Failed (Mismatched symbols or Occur Check failed).
